# Topic 7 · NumPy deep dive (Part 2) — Exercises

Same datasets as the demos (`Data/` folder). 

**Rules of this exercises**
- No Python `for` loops over data rows unless the task says so. Loops over a handful of items (3 species, 4 shops) are fine.
- Each task ends with a **check cell**. Do not edit it; it must run without error.
- Each task has an **explain-why** question. Answer in 1–2 sentences of your own words; these are graded.
- Before you submit: *Kernel → Restart & Run All*.

> Before submitting, **rename this file** to:
>
> **`MSSV_NumpyE2.ipynb`**
>
> where `MSSV` is your student ID (mã số sinh viên). For example, if your student ID is `23120001`, the file name must be `23120001_NumpyE2.ipynb`.
>
> - Use your **student ID only**: no name, no spaces, no extra characters.
> ⚠️ Files with a wrong name may not be graded.

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)
np.set_printoptions(suppress=True, precision=4)

grades = np.genfromtxt("Data/P4DS-Grades.csv", delimiter=",", skip_header=1)
X = grades[:, :4]                                   # HW1, HW2, HW3, Project
official = grades[:, 4]
weights = np.array([0.5 / 3, 0.5 / 3, 0.5 / 3, 0.5])
final = X @ weights
grades.shape

(45, 5)

---
# Part B — Broadcasting, floating point, applications

## B1. Broadcasting: predict the shape

Fill in `predictions` with the result shape as a tuple, or the string `"error"`. **Do it by hand first** using the three rules, then run the check.

In [2]:
pairs = [
    ((4, 3),    (3,)),
    ((4, 3),    (4,)),
    ((4, 1),    (3,)),
    ((5, 1, 3), (4, 3)),
    ((2, 3),    (3, 2)),
    ((10, 1),   (1, 10)),
    ((8, 1, 6, 1), (7, 1, 5)),
    ((3,),      ()),
]

In [3]:
predictions = [
    (4, 3),   # (4, 3) with (3,)
    "error",   # (4, 3) with (4,)
    (4, 3),
    (5, 4, 3),
    "error",
    (10, 10),
    (8, 7, 6, 5),
    (3,),
]

In [4]:
# CHECK (do not edit)
for (sa, sb), guess in zip(pairs, predictions):
    try:
        truth = np.broadcast_shapes(sa, sb)
    except ValueError:
        truth = "error"
    mark = "✅" if guess == truth else "❌"
    print(f"{mark} {str(sa):>14} with {str(sb):<10} -> truth {truth}, you said {guess}")

✅         (4, 3) with (3,)       -> truth (4, 3), you said (4, 3)
✅         (4, 3) with (4,)       -> truth error, you said error
✅         (4, 1) with (3,)       -> truth (4, 3), you said (4, 3)
✅      (5, 1, 3) with (4, 3)     -> truth (5, 4, 3), you said (5, 4, 3)
✅         (2, 3) with (3, 2)     -> truth error, you said error
✅        (10, 1) with (1, 10)    -> truth (10, 10), you said (10, 10)
✅   (8, 1, 6, 1) with (7, 1, 5)  -> truth (8, 7, 6, 5), you said (8, 7, 6, 5)
✅           (3,) with ()         -> truth (3,), you said (3,)


❓ **Explain why:** `(4, 3)` with `(4,)` fails even though both have a 4. How do you make it work if you meant 'one value per row'?

> *Your answer:* When making calculation between (4, 3) and (4, ), the (4, ) array becomes (1, 4) by the first rule and then it checks the size in each dimension. Here, it recognizes that the second dimension having the size 4 while that of other is 3 and due to the second rule which states when one dimension is not equal, the size of one must be 1 for expanding, the calculation between two arrays fails. 'One value per row' means that I use reshape method to change the shape of (4, ) array to (4, 1). Thanks to that, the calculation can be made and the shape of result array is going to be (4, 3).

## B2. Iris: a nearest-centroid classifier

**The data.** `Iris.csv` has 150 iris flowers from 3 species (*setosa*, *versicolor*, *virginica*), 50 of each. Each flower has 4 measurements in cm: sepal length, sepal width, petal length and petal width.

**The question.** Given the 4 measurements of a new flower, can we guess its species?

**The idea: nearest centroid.**
1. For each species, compute its **centroid**: the average flower, i.e. the mean of each measurement over all flowers of that species. This gives 3 "typical" flowers.
2. To classify a new flower, measure its distance to each of the 3 centroids and pick the species whose centroid is **closest**.

We use the squared Euclidean distance between flower $a$ and centroid $c$:

$$d^2(a, c) = \sum_{j=1}^{4} (a_j - c_j)^2$$

**Already done for you** in the cell below:
- `labels`: species as integer codes 0, 1, 2, and `names`: the species names.
- `scaled`: every measurement min–max scaled to $[0, 1]$, so that no single measurement dominates the distance.
- A random split into **120 training flowers** (`X_train`, `y_train`), used to compute the centroids, and **30 test flowers** (`X_test`, `y_test`), used to check the predictions.

**Your tasks** (no loops over flowers; a loop over the 3 species is fine):
1. `centroids`, shape `(3, 4)`: row `c` is the mean of the training flowers of species `c`.
2. `dist_sq`, shape `(30, 3)`: `dist_sq[i, c]` is the squared distance from test flower `i` to centroid `c`, computed with **one broadcasting expression**.
   *Hint:* give `X_test` shape `(30, 1, 4)` and `centroids` shape `(1, 3, 4)`. Subtracting them gives `(30, 3, 4)`, and summing over the last axis gives `(30, 3)`.
3. `y_hat`: the predicted species of each test flower, i.e. the column with the smallest distance in each row. Then `accuracy`: the fraction of test flowers predicted correctly.
4. The **confusion matrix** `conf`, shape `(3, 3)`, where `conf[i, j]` is the number of test flowers of true species `i` predicted as species `j`. A perfect classifier has numbers only on the diagonal. Use a single `np.bincount` or `np.add.at` call.
   *Hint:* encode each pair `(i, j)` as the single number `3*i + j`.


In [5]:
features = np.genfromtxt("Data/Iris.csv", delimiter=",", skip_header=1, usecols=range(4))
species = np.genfromtxt("Data/Iris.csv", delimiter=",", skip_header=1, usecols=4, dtype=str)
names, labels = np.unique(species, return_inverse=True)

lo, hi = features.min(axis=0), features.max(axis=0)
scaled = (features - lo) / (hi - lo)

split_rng = np.random.default_rng(42)
perm = split_rng.permutation(len(scaled))
X_train, y_train = scaled[perm[30:]], labels[perm[30:]]
X_test, y_test = scaled[perm[:30]], labels[perm[:30]]

In [6]:
centroids = np.array([X_train[y_train == label_id].mean(axis=0) 
                      for label_id in range(len(names))])     # (3, 4)
dist_sq =  ((X_test.reshape(30,1,4) - centroids)**2).sum(axis=2)     # (30, 3)
y_hat = dist_sq.argmin(axis=1)
accuracy = ((y_hat == y_test).sum())/y_test.size
conf = np.bincount(3*y_test + y_hat, minlength=9).reshape(3,3)          # bonus, (3, 3)
conf

array([[10,  0,  0],
       [ 0,  8,  1],
       [ 0,  1, 10]])

In [7]:
# CHECK (do not edit)
assert np.shape(centroids) == (3, 4) and np.shape(dist_sq) == (30, 3)
assert np.allclose(centroids[0], X_train[y_train == 0].mean(axis=0))
assert np.isclose(dist_sq[0, 1], ((X_test[0] - centroids[1]) ** 2).sum())
assert 0 <= accuracy <= 1
print("B2 OK, accuracy", accuracy)

B2 OK, accuracy 0.9333333333333333


❓ **Explain why:** To classify a new flower, this method compares it with only 3 centroids, not with all 120 training flowers. What does this save, and in what kind of data would comparing with every training flower give better predictions?

> *Your answer:* Comparing a new flower with three centroids reduces the number of distance calculations from 120 to 3 per prediction. Once the centroids are computed, the classifier only needs to store those three centroids to make predictions. Comparing the new flower with all the training samples can give better predictions when a species forms several separate clusters. Its centroid may lie between those clusters, far from most flowers of that species. In that case, a new flower may be close to training flowers of its own species even if it is far from its species’ centroid.

## B3. Find and fix the bugs

A teammate wrote this report function. It runs without errors, but **four** results are wrong.

In [8]:
def buggy_report(X, official, prices_million, photo):
    # 1. error of our grade formula against the official grades
    pred = X @ weights.reshape(-1, 1)
    mse = np.mean((official - pred) ** 2)

    # 2. how many students got exactly 8.21 officially, according to our formula?
    n_821 = (X @ weights == 8.21).sum()

    # 3. convert prices given in millions to whole đồng
    dong = (prices_million * 1_000_000).astype(np.int64)

    # 4. brighten a uint8 photo by 80
    bright = photo + 80
    return mse, n_821, dong, bright

prices_million = np.array([8.19, 12.99, 7.5, 4.59])
photo = np.array([[10, 200], [180, 250]], dtype=np.uint8)
buggy_report(X, official, prices_million, photo)

(6.5711410699588475,
 0,
 array([ 8189999, 12990000,  7500000,  4590000]),
 array([[90, 24],
        [ 4, 74]], dtype=uint8))

In [9]:
def fixed_report(X, official, prices_million, photo):
    # YOUR CODE HERE: copy buggy_report and fix the four bugs
    
    # 1. error of our grade formula against the official grades
    pred = X @ weights
    mse = np.mean((official - pred) ** 2)

    # 2. how many students got exactly 8.21 officially, according to our formula?
    n_821 = (np.isclose(pred, 8.21, atol=0.005)).sum()

    # 3. convert prices given in millions to whole đồng
    dong = (prices_million * 1_000_000).round().astype(np.int64)

    # 4. brighten a uint8 photo by 80
    bright = (photo + np.minimum((255-photo),80)).astype(np.uint8)
    return mse, n_821, dong, bright
fixed_report(X, official, prices_million, photo)

(7.407407407407848e-06,
 1,
 array([ 8190000, 12990000,  7500000,  4590000]),
 array([[ 90, 255],
        [255, 255]], dtype=uint8))

In [10]:
# CHECK (do not edit)
mse, n_821, dong, bright = fixed_report(X, official, prices_million, photo)
assert mse < 1e-4, "bug 1 not fixed"
assert n_821 == (official == 8.21).sum(), "bug 2 not fixed"
assert dong.tolist() == [8_190_000, 12_990_000, 7_500_000, 4_590_000], "bug 3 not fixed"
assert bright.tolist() == [[90, 255], [255, 255]], "bug 4 not fixed"
print("B3 OK")

B3 OK


❓ **Explain why:** For bug 3, `int()` and `.astype(int)` give 8,189,999 for 8.19 million. Explain what happens, step by step.

> *Your answer:* The floating point number 8.19 can not be represented exactly in binary floating-point format, so it is going to be 8.189999...99. When carrying out the calculation 8.19 * 1_000_000, the raw result is 8,189,999.99999...99 and after integer casting either through int() or .astype(int), it becomes 8,189,999 due to truncation of fractional part.

## B4. kNN without running out of memory

### Background: the k-nearest-neighbours (kNN) classifier

To predict the class of a new point, kNN:
1. computes the distance from the new point to **every** labelled (training) point,
2. takes the **k** closest training points (its "neighbours"),
3. predicts the class that appears most often among those k neighbours (**majority vote**).

The provided function `knn_predict_plain` does this for all test points at once with broadcasting:

```python
difference = X_test.reshape(len(X_test), 1, -1) - X_train   # keeps n_test rows, any number of features
```

### The problem

`diff` holds one row for **every pair** (test point, training point). Its size grows with `n_test × n_train × d`:

| n_test | n_train | d | memory for `diff` |
|---|---|---|---|
| 30 | 120 | 4 | 0.1 MB |
| 5,000 | 15,000 | 2 | **1.2 GB** |
| 20,000 | 100,000 | 4 | **64 GB** |

On the data below (15,000 training points, 5,000 test points), the plain version needs about 1.2 GB just for `diff`. A bit more data and it will not fit in memory.

### Your task

Write `knn_predict_chunked(X_train, y_train, X_test, k=5, chunk=500)`, which gives the **same predictions** as `knn_predict_plain` but never builds the big `diff` array. Use two ideas:

**Idea 1: the expanded distance formula.** Just like $(a - b)^2 = a^2 + b^2 - 2ab$, for vectors:

$$\lVert a - b\rVert^2 = \lVert a\rVert^2 + \lVert b\rVert^2 - 2\,a\cdot b$$

For a block of test points `Bc` (shape `(c, d)`) and all training points `A` (shape `(n_train, d)`):

```python
d2 = (Bc ** 2).sum(axis=1)[:, None] + (A ** 2).sum(axis=1)[None, :] - 2 * Bc @ A.T   # (c, n_train)
```

This only needs a `(c, n_train)` matrix and one matrix multiplication, never a `(c, n_train, d)` array.

**Idea 2: process the test points in chunks.** Loop over the test set `chunk` rows at a time (e.g. 500), compute the distances for that block, find its neighbours, store its predictions, and move on. Only one `(chunk, n_train)` matrix exists at any time: 500 × 15,000 × 8 bytes = 60 MB instead of 1.2 GB.

> A loop over **chunks** is fine. A loop over **single points** is not.

**Two floating-point safeguards** (the expanded formula subtracts large, nearly equal numbers):
- **Centre the data first:** subtract the mean of `X_train` from both `X_train` and `X_test`. Distances do not change, because every point shifts by the same amount, but the numbers become small and the subtraction stays accurate.
- **Clip at 0:** `np.maximum(d2, 0)`. Rounding can still produce tiny negative "squared distances".

### Steps (follow the `...` in the skeleton)

1. Centre `X_train` and `X_test` using the mean of `X_train` → `A`, `B`.
2. Compute the squared length of every training point once: `sq_train`, shape `(n_train,)`.
3. For each chunk `Bc` of `B`:
   - a. `d2`: squared distances with the expanded formula, shape `(c, n_train)`
   - b. clip `d2` at 0
   - c. `nearest`: indices of the k smallest values in each row, shape `(c, k)` (hint: `np.argpartition`)
   - d. majority vote: count each class among the neighbours' labels and take the most frequent one (hint: copy the vote line from `knn_predict_plain`)

In [11]:
# 20,000 points in 2D, 3 classes, each class a cloud around its own centre
centres = np.array([[0, 0], [3, 0], [0, 3]])
y_all = rng.integers(0, 3, 20_000)
X_all = centres[y_all] + rng.normal(0, 1, (20_000, 2))

X_train, y_train = X_all[:15_000], y_all[:15_000]     # 15,000 labelled points
X_test,  y_test  = X_all[15_000:], y_all[15_000:]     #  5,000 points to classify

print(X_train.shape, X_test.shape)

(15000, 2) (5000, 2)


In [12]:
# Plain kNN
def knn_predict_plain(X_train, y_train, X_test, k=5):
    "kNN with the full (n_test, n_train, d) difference array. Fine for small data only."
    # Shape: (n_test, 1, d) - (n_train, d) -> (n_test, n_train, d)
    difference = X_test.reshape(len(X_test), 1, -1) - X_train

    # Shape: (n_test, n_train, d) -> (n_test, n_train)
    dist_sq = (difference ** 2).sum(axis=2)

    # Shape: (n_test, n_train) -> (n_test, k)   indices of the k nearest training points
    nearest = np.argpartition(dist_sq, k, axis=1)[:, :k]

    # Shape: (n_test, k)   labels of those neighbours
    neighbour_labels = y_train[nearest]

    # Shape: (n_test, k, 1) == (3,) -> (n_test, k, 3) -> sum over k -> (n_test, 3)   votes per class
    votes = (neighbour_labels.reshape(len(X_test), -1, 1) == np.arange(3)).sum(axis=1)

    # Shape: (n_test, 3) -> (n_test,)   class with the most votes
    return votes.argmax(axis=1)

# How much memory would the plain version need on the full data?
gb = len(X_test) * len(X_train) * X_train.shape[1] * 8 / 1e9
print(f"plain kNN on the full data would need about {gb:.1f} GB for diff alone")

plain kNN on the full data would need about 1.2 GB for diff alone


In [13]:
# YOUR CODE HERE
def knn_predict_chunked(X_train, y_train, X_test, k=5, chunk=500):
    n_classes = y_train.max() + 1

    # Step 1: centre both sets using the mean of the TRAINING data
    A = X_train - X_train.mean(axis=0)            # X_train - mean
    B = X_test - X_train.mean(axis=0)            # X_test  - mean

    # Step 2: squared length of every training point, computed once
    sq_train = (A**2).sum(axis=1)     # shape (n_train,)

    y_hat = np.empty(len(B), dtype=int)

    # Step 3: loop over CHUNKS of test points (not over single points)
    for start in range(0, len(B), chunk):
        Bc = B[start:start + chunk]                     # (c, d), c <= chunk

        # 3a. squared distances with the fast formula, shape (c, n_train)
        d2 = (Bc**2).sum(axis=1)[:, None] + sq_train[None, :] - 2 * Bc @ A.T

        # 3b. clip small negative values caused by rounding
        d2 = np.maximum(d2,0)

        # 3c. indices of the k nearest training points, shape (c, k)
        nearest = np.argpartition(d2,kth=k,axis=1)[:,:k]

        # 3d. majority vote -> one label per test point, shape (c,)
        neighbour_labels = y_train[nearest]                     # (c, k)
        votes = (neighbour_labels[:, :, None] == np.arange(3)).sum(axis=1)
        y_hat[start:start + chunk] = votes.argmax(axis=1)

    return y_hat

In [14]:
t0 = time.perf_counter()
y_hat = knn_predict_chunked(X_train, y_train, X_test, k=7)
print(f"time: {time.perf_counter() - t0:.2f} s | accuracy: {(y_hat == y_test).mean():.2%}")

time: 1.69 s | accuracy: 89.46%


In [15]:
# On a small subset, the chunked version must agree with the plain version
small_plain = knn_predict_plain(X_train[:2000], y_train[:2000], X_test[:500], k=7)
small_chunk = knn_predict_chunked(X_train[:2000], y_train[:2000], X_test[:500], k=7, chunk=128)
agreement = (small_plain == small_chunk).mean()
print(f"agreement with the plain version: {agreement:.1%}")
assert agreement > 0.99, "your chunked kNN disagrees with the plain version"
assert y_hat.shape == y_test.shape
print("Challenge OK")

agreement with the plain version: 100.0%
Challenge OK
